<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Productos identificados por AmCham y su margen de oportunidad

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.

In [1]:
import pandas as pd
import numpy as np

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import raw, temp, tables

### Parámetros

Define el umbral de proximidad en el espacio de producto que separa las oportunidades a la
vista de las de largo alcance.

In [3]:
cercania = 0.65

valor_minimo = 50   # miles de dólares

### Partidas con ventaja comparativa revelada

Extrae del estudio de AmCham las partidas arancelarias a cuatro dígitos con ventaja
comparativa revelada mayor que uno.

**Nota de auditoría:** el notebook original asigna `arr_can` dos veces seguidas, con la hoja
`Table 2` y luego con la hoja `Table 3`. La segunda asignación sobrescribe la primera, de modo
que las partidas de `Table 2` nunca entran al conjunto. El código se migra tal cual para no
alterar el resultado. Ver `tools/markdown/AUDITORIA_MIGRACION.md`.

In [4]:
dict_dfs = pd.read_excel(raw / 'amcham' / 'AmChamColombia-EEUU-Oportunidades-EstudioVCR 2025 copia.xlsx', sheet_name=None, header=1)

arr_chn = dict_dfs['Table 1'].dropna(subset=['Producto']).dropna().query("`VCR>1` > 1")['Partida'].unique()
arr_can = dict_dfs['Table 2'].dropna(subset=['Producto']).dropna().query("`VCR>1` > 1")['Partida'].unique()
arr_can = dict_dfs['Table 3'].dropna(subset=['Producto']).dropna().query("`VCR>1` > 1")['Partida'].unique()

combined_set = set(arr_can) | set(arr_chn) | set(arr_can)

formatted_list = list(map(lambda x: str(x).zfill(4), combined_set))

print(f"partidas HS4 identificadas: {len(formatted_list)}")

partidas HS4 identificadas: 53


### Traducción a la nomenclatura HS 2012

Cruza las partidas de AmCham, expresadas en HS 2022, con la tabla de correspondencia hacia
HS 2012, y conserva los códigos de seis dígitos correspondientes.

In [5]:
df_hs22_12 = pd.read_excel(raw / 'comercio' / 'correlativas' / 'HS2022toHS2012ConversionAndCorrelationTables.xlsx')
df_hs22_12['From HS 2022'] = df_hs22_12['From HS 2022'].astype(str).str.zfill(6)
df_hs22_12['To HS 2012'] = df_hs22_12['To HS 2012'].astype(str).str.zfill(6)
df_hs22_12['From HS4 2022'] = df_hs22_12['From HS 2022'].astype(str).str[:4]

df_AmCham = df_hs22_12.query("`From HS4 2022` in @formatted_list").reset_index(drop=True).drop_duplicates().copy()

### Intersección con la lista de productos priorizados

Cruza los productos de AmCham con los productos únicos del ejercicio de caracterización para
identificar la intersección entre ambas listas.

In [6]:
df_fede = pd.read_excel(raw / 'exportaciones' / 'productos' / '2 - Output Final 040725 V02 - EXT.xlsx', sheet_name='Productos únicos')[['HS6']].map(lambda x: str(x).zfill(6)).drop_duplicates().reset_index(drop=True)

df_interseccion = df_fede.merge(df_AmCham, left_on='HS6', right_on='To HS 2012', how='inner')

print(f"productos AmCham: {df_AmCham['To HS 2012'].nunique()}   en la intersección: {df_interseccion['HS6'].nunique()}")

productos AmCham: 284   en la intersección: 32


### Composición por industria

Agrupa los productos de AmCham y los de la intersección por industria CIIU a tres dígitos, y
exporta ambas tablas en un mismo libro.

In [7]:
df_ciiu = pd.read_parquet(temp / 'df_ciiu.parquet')
df_ciiu['CIIU'] = df_ciiu['CIIU'].astype(str).str[:3]
df_ciiu = df_ciiu.drop_duplicates()

df_ind_amcham = df_AmCham[['To HS 2012']].drop_duplicates().merge(df_ciiu, left_on='To HS 2012', right_on='HS6', how='left').groupby(['CIIU','Industria'])['HS6'].nunique().reset_index().sort_values('HS6', ascending=False)
df_ind_interseccion = df_interseccion[['HS6']].drop_duplicates().merge(df_ciiu, left_on='HS6', right_on='HS6', how='left').groupby(['CIIU','Industria'])['HS6'].nunique().reset_index().sort_values('HS6', ascending=False)

tables.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(tables / 'AmChamOverLap.xlsx', engine='xlsxwriter') as writer:
    df_ind_amcham.to_excel(writer, sheet_name='Ind AmCham', index=False)
    df_ind_interseccion.to_excel(writer, sheet_name='Ind Intersección AmCham Fede', index=False)

### Clasificación por margen de oportunidad

Asigna a cada producto de AmCham su margen de oportunidad.

**Pasos realizados:**

1. **Oportunidades a profundizar:** los productos que Colombia ya exporta por encima del umbral de valor.

2. **Vecindad en el espacio de producto:** para cada producto exportado, separa los productos cercanos de los lejanos según el umbral de proximidad.

3. **Asignación del margen restante:** los productos sin margen asignado se clasifican como oportunidades a la vista si son cercanos a algún producto exportado, y de largo alcance en caso contrario.

**Nota de auditoría:** en el bucle de asignación la variable `tuple_margen` conserva el valor de
la iteración anterior cuando un producto no cae en ninguna de las dos condiciones, lo que puede
duplicar la clasificación de otro producto. El código se migra tal cual.

In [8]:
df_comercio_col = pd.read_parquet(temp / 'df_comercio_mundial.parquet').query("PartnerISO3 == 'COL' and `TradeValue in 1000 USD` >= @valor_minimo")[['ProductCode']].rename(columns={'ProductCode': 'HS6'}).drop_duplicates()
df_comercio_col['HS6'] = df_comercio_col['HS6'].astype(str).str.zfill(6)
df_comercio_col['Margen'] = 'Oportunidades a profundizar'
list_comercio_col = df_comercio_col['HS6'].tolist()

df_product_space = pd.read_parquet(temp / 'df_product_space.parquet')
df_product_space_col = df_product_space[list_comercio_col].copy()

df_amcham = df_AmCham[['To HS 2012']].drop_duplicates().merge(df_ciiu, left_on='To HS 2012', right_on='HS6', how='left').drop(columns=['To HS 2012','CIIU'])
df_amcham = df_amcham.merge(df_comercio_col, left_on='HS6', right_on='HS6', how='left')

dict_productos_cerca = dict()
dict_productos_lejos = dict()
for producto in df_product_space_col:
    dict_productos_cerca[producto] = df_product_space_col[producto].drop(producto).reset_index().query(f"`{producto}`>={cercania}")['index'].to_list()
    dict_productos_lejos[producto] = df_product_space_col[producto].drop(producto).reset_index().query(f"`{producto}`<{cercania}")['index'].to_list()
dict_productos_cerca = {k: v for k, v in dict_productos_cerca.items() if v}
dict_productos_lejos = {k: v for k, v in dict_productos_lejos.items() if v}

list_margen = list()

for productos in df_amcham[df_amcham.isnull().any(axis=1)]['HS6'].unique().tolist():
    for producto_lejos in dict_productos_lejos.keys():
        if productos in dict_productos_lejos[producto_lejos]:
            tuple_margen = (productos,'Oportunidades de largo alcance')
    for producto_cerca in dict_productos_cerca.keys():
        if productos in dict_productos_cerca[producto_cerca]:
            tuple_margen = (productos,'Oportunidades a la vista')
    list_margen.append(tuple_margen)

df_amcham = df_amcham.merge(pd.DataFrame(list_margen,columns=['HS6','Margen']),how='left',on='HS6').drop_duplicates().reset_index(drop=True)
df_amcham['Margen'] = df_amcham['Margen_x'].fillna(df_amcham['Margen_y'])
df_amcham = df_amcham.drop(columns=['Margen_x','Margen_y']).dropna().reset_index(drop=True)

temp.mkdir(parents=True, exist_ok=True)
df_amcham.to_parquet(temp / 'df_amcham.parquet', index=False)
print(f"productos clasificados: {len(df_amcham):,}")

productos clasificados: 284
